# 步骤 5：固定强度 vs 连续自适应（三档分格测）

按 **暗光 → 逆光 → 常规** 的顺序往下跑，**一格测一档**，每格约 30 秒（倒计时 10 秒 + 抓 3 帧）。
这样测完一档可以立刻调下一档的灯，不用一气呵成。结果会**合并**写进 `data/step5_compare.csv`。

看什么算通过：每条结果里**不出现** `⚠ 摆位没到位`（即 f 到目标中心的距离 < 20）。

摆位目标（`data/features.csv` 的实测中心）：

| 档位 | f=(luma, luma2, grad) | 场景 |
|------|----------------------|------|
| 暗光 | (9.7, 0.2, 17.2) | 主光源全关 / 拧到最暗档 |
| 逆光 | (66.4, 44.6, 21.4) | 后壁光板开 + **哑光黑色遮挡物**（亮暗并存，画面别带纹理） |
| 常规 | (96.5, 68.6, 29.9) | 主光源中亮 + 对着打印网格/文字卡 |

> 为什么过曝只出现在 `mode0 固定 ×1.5` 那一行：过曝只会发生在**增益**这一路，
> 而 mode 0 平时只在暗光档被选中。报告里那对数字就看 `mode0` 那两行。


In [ ]:
import os, sys

ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

import tools.step5_compare as s5
print("工程根目录:", ROOT)
print("新版脚本:", hasattr(s5, "_acquire_pl"), "（必须是 True）")
done = sorted(s5._read_csv_rows(os.path.join(ROOT, "data", "step5_compare.csv")))
print("CSV 里已有的档位:", done if done else "（空）")


## 第 1 格：暗光档

**主光源全关**（或拧到最暗档），目标 `f=(10, 0, 17)`。
跑之前看一眼画面：漏光会把暗光档顶到 `f_luma` 20 以上。


In [ ]:
s5.main(["--only", "0", "--wait", "10", "--shots", "3"])


## 第 2 格：逆光档

**后壁光板开、主光源关**，前面放**哑光黑色的遮挡物**（占画面 1/3~1/2，四周露出亮光板）。
目标 `f=(66, 45, 21)`：`f_luma2` 要上去（亮暗并存），`f_grad` 别超 25（画面别带纹理）。

- `f_luma2` 上不去 → 光板再打亮一些 / 黑物再放大一些；
- `f_grad` 超标 → 换素面的黑物（别带字迹、logo、褶皱），或给光板加柔光。


In [ ]:
s5.main(["--only", "1", "--wait", "10", "--shots", "3"])


## 第 3 格：常规档

**主光源开到中亮**，镜头对着打印的网格/文字卡，目标 `f=(96, 69, 30)`。
「固定过曝 → 自适应不过曝」那对数字就来自这一档，务必让摆位达标再抄数。


In [ ]:
s5.main(["--only", "2", "--wait", "10", "--shots", "3"])


## 汇总：把三档合并后看一眼

`report()` 会把 `data/step5_compare.csv` 读回来打印成表，并告诉你：还差哪档没测、
哪几行摆位没达标（**只有「摆位 = OK」的行才能进报告**）。


In [ ]:
s5.report()


## 收尾（可选）

`release()` = 停通道 + 还 DMA 缓冲 + 清缓存。

- 要把 PL 让给别的 notebook / 后台脚本时跑它；只接着用这一个 notebook 就不用跑；
- 遇到 `Allocate failed` 也先跑它（比重启板子快，脚本自己也会先自救一次）。


In [ ]:
s5.release()
